# 🔬 Science Paper Analyzer
## Colab Notebook — Анализ научных статей на достоверность

Программа собирает статьи по вашей теме из **10+ научных источников**
и анализирует их по трём критериям:
- **📊 Цитируемость** — количество ссылок на статью
- **📝 Текст** — детектор AI-генерации (эвристики)
- **🏛️ Журнал** — проверка по Beall's List (1200+ хищнических журналов)

---

## Шаг 1. Клонирование репозитория и установка зависимостей

> **Просто нажмите ▶ (Run) на этой ячейке.**

Скачает код с GitHub и установит все нужные библиотеки.

In [ ]:
import subprocess, sys, os, importlib

REPO_URL = "https://github.com/DmitPerson42/science-paper-analyzer.git"
REPO_DIR = "science-paper-analyzer"

# Всегда свежий код: удаляем старый и клонируем заново
if os.path.exists(REPO_DIR):
    import shutil
    shutil.rmtree(REPO_DIR, ignore_errors=True)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True,
               capture_output=True, text=True)
os.chdir(REPO_DIR)

# Ставим зависимости (только тех, что нет)
required = ["requests", "pandas", "openpyxl", "lxml",
            "beautifulsoup4", "numpy", "scikit-learn"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing,
                   check=True, capture_output=True, text=True)

print("✅ Шаг 1 выполнен! Переходите к Шагу 2.")

## Шаг 2. Импорт и запуск анализа

Просто запустите ячейку ниже — всё сделается автоматически.

In [ ]:
# ============================================================
# ВСЁ В ОДНОЙ ЯЧЕЙКЕ — запуск + анализ + результаты
# ============================================================
import sys, pandas as pd
sys.path.insert(0, ".")

from analyzer import PaperAnalyzer

# ------ НАСТРОЙКА ------
QUERY = "filtering generative text data language model collapse"
MAX_PER_SOURCE = 5
# -----------------------

print(f"📌 Тема: {QUERY}")
print(f"📌 Статей/источник: {MAX_PER_SOURCE}\n")

# Создаём анализатор (в консольном режиме)
analyzer = PaperAnalyzer(verbose=True)

# --- СБОР ---
print("="*50)
print("📡 СБОР СТАТЕЙ")
print("="*50)
papers = analyzer.collect_papers(QUERY, max_per_source=MAX_PER_SOURCE)

if not papers:
    print("\n❌ Статей не найдено. Попробуйте другую тему.")
    df = pd.DataFrame(columns=["title","authors","year","source",
                               "url","abstract","citation_score",
                               "text_score","journal_score",
                               "overall_score","verdict"])
else:
    # --- АНАЛИЗ ---
    print("\n" + "="*50)
    print("🔬 АНАЛИЗ СТАТЕЙ")
    print("="*50)
    df = analyzer.analyze_papers(papers)
    print(f"\n✅ Проанализировано {len(df)} статей.\n")

    # --- ТАБЛИЦА ---
    display_cols = ["title", "authors", "year", "source",
                    "overall_score", "verdict"]
    df_view = df[display_cols].copy()
    df_view.columns = ["📖 Название", "👤 Авторы", "📅 Год",
                       "📡 Источник", "Score", "⚖️ Вердикт"]
    display(df_view)

    # --- СТАТИСТИКА ---
    total = len(df)
    rc = len(df[df["verdict"] == "Real"])
    sc = len(df[df["verdict"] == "Suspicious"])
    fc = len(df[df["verdict"] == "Fake"])

    print("\n" + "="*50)
    print("           📊 СВОДКА")
    print("="*50)
    print(f"  Всего:     {total}")
    print(f"  ✅ Real:   {rc} ({rc/total*100:.1f}%)")
    print(f"  ⚠️ Sus:    {sc} ({sc/total*100:.1f}%)")
    print(f"  ❌ Fake:   {fc} ({fc/total*100:.1f}%)")
    print("="*50)

    if sc > 0 or fc > 0:
        print("\n🔍 Статьи, требующие внимания:")
        bad = df[df["verdict"] != "Real"]
        display(bad[display_cols])

print("\n✅ Анализ завершён!")

## Шаг 3. Скачать результаты (CSV / Excel)

Запустите ячейку ниже, чтобы скачать файлы.

In [ ]:
if "df" in dir() and len(df) > 0:
    csv_path = "analysis_results.csv"
    xlsx_path = "analysis_results.xlsx"

    analyzer.export_csv(df, csv_path)
    analyzer.export_excel(df, xlsx_path)

    from google.colab import files
    files.download(csv_path)
    print("✅ CSV скачан!")
else:
    print("⚠️ Нет данных для экспорта. Сначала выполните Шаг 2.")

---

### 📚 О системе

**Science Paper Analyzer v1.0**

**Источники:** arXiv, Semantic Scholar, OpenReview, ACL Anthology,
JMLR, ResearchGate, CyberLeninka, eLibrary, Dissercat, FIPS

**Критерии:** цитируемость, AI-детектор текста, Beall's List

**Формат вывода:** CSV / Excel (с цветовой маркировкой)

**🔗 GitHub:** https://github.com/DmitPerson42/science-paper-analyzer